# Visual Analysis Dashboard — run on Colab with a GPU

Runs the whole product on one Colab GPU machine and gives you a public URL to open it.

The backend serves the dashboard itself, so there is **one port and one address** —
no CORS, no second server, nothing to reconfigure when the tunnel hostname changes.

This is the whole product in one place: the ten monitoring capabilities, the
training desk, and the **AI Safety Lab** — a simulated factory floor under
*Training → Lab*, which teaches the detection pipeline the other pages run for
real. The Lab needs no camera, no GPU and no backend of its own, so it works
from the moment the page loads, including on any module still downloading its
weights.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Run the cells in order. Roughly 4–5 minutes the first time.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo "NO GPU — set Runtime > Change runtime type > T4 GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: running on CPU. It will work, but inference will be slow.")

## 2. Get the code

The repository is private, so this needs a GitHub token.

Create a **fine-grained** token at
github.com → Settings → Developer settings → Personal access tokens →
Fine-grained tokens:

- **Repository access:** Only select repositories → this repo
- **Permissions:** Repository permissions → **Contents: Read-only**
- **Expiration:** whatever suits — 7 days is plenty for testing

That is the least a clone needs. It cannot push, cannot touch other repos, and
expires on its own.

The token is read with `getpass`, kept in this session's environment only, and
handed to git through a credential helper — so it is never written into the
notebook, never stored in `.git/config`, and disappears when the runtime does.


In [ ]:
import getpass, os, subprocess

REPO   = "rootstocktechai-debug/vikasgroup_visual_analytics_fullstack_beta"
BRANCH = "palak"

token = getpass.getpass("GitHub token (input hidden): ").strip()

import shutil
if os.path.exists("/content/app"):
    shutil.rmtree("/content/app")

r = subprocess.run(
    ["git","clone","--depth","1","--branch",BRANCH,
     f"https://{token}@github.com/{REPO}.git","/content/app"],
    capture_output=True, text=True)

print("cloned" if r.returncode == 0 else r.stderr[-800:])
del token

!ls /content/app/backend/models/

## 3. Install the backend

Colab already ships a CUDA build of torch, so this deliberately does **not**
install from `requirements.txt` — that pins a CPU torch and would replace the
GPU one. Only the packages Colab is missing are installed.

`insightface` powers Face Recognition, and also the head check Face Masks
makes before it accuses anybody — masks load the face *finder* only, without
the much larger model that answers who somebody is. One pack (~280 MB) serves
both, downloaded automatically the first time whichever page needs it is
opened. The GPU
build of `onnxruntime` is used so it runs on Colab's GPU rather than
its two CPU cores — the difference is about 400ms a frame. If the GPU
build cannot load, the backend falls back to the CPU and says so.

In [ ]:
!pip install -q fastapi==0.115.6 "uvicorn[standard]==0.34.0" python-multipart==0.0.20 ultralytics==8.3.58 insightface==1.0.1 openpyxl==3.1.5 onnxruntime-gpu==1.28.0 2>&1 | tail -2
# Burned-in timestamp OCR. --no-deps is load-bearing: rapidocr's own
# dependency list would drag a CPU onnxruntime over the GPU build above
# and a bare opencv-python over Colab's contrib one.
!pip install -q pyclipper==1.3.0.post6 shapely==2.0.7 2>&1 | tail -1
!pip install -q --no-deps rapidocr-onnxruntime==1.4.4 2>&1 | tail -1
import torch
print("CUDA still available after install:", torch.cuda.is_available())

## 4. Build the dashboard

This builds the Lab too. Its source lives in `lab/`, and the dashboard imports
it rather than keeping a copy — one implementation of the simulation, one set
of tests over it. There is nothing extra to install for it: the build resolves
React and the icon set from this app's own `node_modules`, so `lab/` needs no
`npm install` of its own.

In [ ]:
%%bash
cd /content/app/frontend
node --version
npm install --no-audit --no-fund --silent 2>&1 | tail -2
npm run build 2>&1 | tail -4
ls -la dist/index.html

## 5. Start the server

Runs on port 8000, serving both the API and the built dashboard.
Logs go to `/content/server.log`.

In [ ]:
import subprocess, time, urllib.request, pathlib, json

pathlib.Path("/content/app/backend/storage/uploads").mkdir(parents=True, exist_ok=True)

log = open("/content/server.log", "w")
server = subprocess.Popen(
    ["python","-m","uvicorn","app.main:app",
     "--host","0.0.0.0","--port","8000",
     "--reload","--reload-dir","app"],     # picks up backend edits without a restart
    cwd="/content/app/backend", stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print("server up")
        break
    except Exception:
        time.sleep(2)
else:
    print("server failed to start — last of the log:")
    print(open("/content/server.log").read()[-2000:])

status = json.load(urllib.request.urlopen("http://127.0.0.1:8000/system/status"))["data"]
print("model :", status["model"])
print("modules:", [m["module_id"] for m in
      json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/modules"))["data"]])


## 6. Open it

Two ways. **Try the first** — it is built into Colab, needs no account and no
external service.

In [ ]:
# Option A — Colab's built-in port proxy (recommended)
# Works in the browser session you are signed into. Nothing to install.
from google.colab.output import eval_js
print("OPEN THIS:", eval_js("google.colab.kernel.proxyPort(8000)"))

**Option B — a public URL.** Use this instead if Option A misbehaves, or if you
want to open the dashboard on your phone or send it to a colleague. It publishes
port 8000 on a temporary address that dies with this session.

In [ ]:
# Option B — cloudflared public URL (no account needed)
import re, subprocess, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ["cloudflared","tunnel","--url","http://localhost:8000","--no-autoupdate"],
    stdout=open("/content/tunnel.log","w"), stderr=subprocess.STDOUT)

url = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break

if url:
    print("\n" + "="*66)
    print("  OPEN THIS:", url)
    print("="*66)
else:
    print("Tunnel did not come up. Last lines of /content/tunnel.log:")
    print(open("/content/tunnel.log").read()[-1200:])
    print("\nUse Option A above instead.")

## 7. Give it something to watch

Colab has no webcam, so use a video file. Two options:

**Upload one here**, then pick *Video file* in the dashboard — or run the cell
below to upload and switch to it in one step.

**Or use an IP camera**: pick *Network camera* in the dashboard and enter an
RTSP URL. Note this only works if the camera is reachable from the public
internet — a camera on your plant LAN will not be reachable from Colab.

In [ ]:
from google.colab import files
import shutil, pathlib, urllib.request, json

up = files.upload()
name = next(iter(up))
dest = pathlib.Path("/content/app/backend/storage/uploads")/name
shutil.move(name, dest)

req = urllib.request.Request(
    "http://127.0.0.1:8000/camera/source",
    data=json.dumps({"source": f"storage/uploads/{name}"}).encode(),
    headers={"Content-Type":"application/json"}, method="POST")
print(json.load(urllib.request.urlopen(req)))
print("Now open the dashboard and choose a monitoring page.")

## 8. Use the camera on *your* machine

Colab has no webcam — but your browser does, and the dashboard can use it.

On any monitoring page, pick **This device** and press **Start watching**. The
browser captures from your webcam, sends frames to this Colab machine, the GPU
analyses them, and the annotated picture comes back. The camera never leaves
your laptop; only pictures cross the wire.

**This needs an `https` address**, because browsers refuse camera access over
plain http. Both options in section 6 are https, so either works.

Throughput is shown under the camera panel — typically 5–15 pictures a second
depending on your upload speed and the round trip to Colab.


## 9. Editing while it runs

**Backend** — the server was started with `--reload`, so it restarts itself
whenever a file under `backend/app` changes. Pull new commits and it picks them
up within a second or two.

A pull brings down changes to both halves, but only the backend reacts on its
own: everything under `frontend/` is served from a build, so a change to the
dashboard needs the rebuild cell below and a refresh of the tab.


In [ ]:
!cd /content/app && git pull --ff-only
# uvicorn --reload notices the change and restarts on its own.
import time, urllib.request
time.sleep(4)
print(urllib.request.urlopen("http://127.0.0.1:8000/health").read().decode())


**Frontend** — two choices.

*Rebuild here* (simple, a few seconds per change):


In [ ]:
!cd /content/app/frontend && npm run build 2>&1 | tail -3
print("Rebuilt. Refresh the dashboard tab.")


*Or run the dashboard on your own machine with hot reload* (instant, best while
working on the UI). On your laptop, with the tunnel URL from section 6:

```bash
git clone <repo> && cd repo/frontend && npm install
VITE_BACKEND_ORIGIN=https://<your-tunnel>.trycloudflare.com npm run dev
```

Open <http://localhost:5173>. Vite proxies the API and the camera WebSocket to
Colab, so the GPU does the work while the UI reloads the instant you save a
file. `localhost` is a secure context, so the browser camera works there too.


## 10. What to try

| Page | What to do |
|---|---|
| **Dashboard** | Overall status across every module — what is being watched, and anything that needs attention |
| **Restricted Zone** | *Mark zones* → click corners, double-click to finish each one, mark as many as the floor needs. Click a name in the list to change it — the alarm then says it: “Person is in restricted zone {name}” — and while somebody is inside, the zone shows how long they have been there |
| **Curfew** *(on the Restricted Zone page)* | Hours when nobody should be here at all. *Set a curfew* → a start, an end, and the days it runs. While it is running the **whole camera view** becomes the restricted area — there is no shape to draw, anybody in the picture raises an alert, and the marked zones are not consulted. Outside those hours the zones watch the camera exactly as before. An overnight window is the normal case: tick the day it *starts*, so a Friday-night 22:00–06:00 curfew covers Saturday's small hours with only Friday ticked |
| **Vehicle in Restricted Zone** | *Mark area* → the same drawing, on its own area — a forklift aisle and a people-exclusion floor are usually different shapes, so this one is kept apart from the zone above. A forklift standing on it raises an alarm that says so aloud |
| **Object Blocking Walkways** | *Mark walkway* → the same drawing again, on the lane that has to stay *clear* rather than one to keep off. Put something on the floor inside it — a box, a bag, a chair — and after a few seconds it alarms. Walk through it yourself and it does not: people are cut out of the picture before the floor is judged |
| **Safety Gear** | Helmet and vest checks, per person, with a count of who is missing what |
| **Gloves** | Bare-hand detection — easiest to demo with your own hands |
| **Face Masks** | Checks that face masks are being worn, per person |
| **Face Recognition** | The AI announces registered people by name the moment they appear on any input. Workers join the register automatically through the **Registration** page (their photos are the enrollment) and are shown **green** with an identified-worker card — details, training state, and the Skilled/Unskilled verdict. This page watches and manages the register; it no longer carries its own registration form |
| **Workstation Absence** | *Mark workstations* → drag a box around each place somebody should be, name it. Step out of view and it announces the workstation by name. The page also graphs each station's recent record — when it was manned, idle or not watched, and the manned-against-idle totals with their difference |
| **Suspended load detection** | *Mark area* → draw the floor the lifting machine works over. Somebody standing in it raises an alarm, and the page labels them with the share of their lower body inside the area. This is phase one: it watches the area for **people**, and does **not** detect the load. The page lists all six questions the capability will answer and marks which one it can answer today |
| **Cannot check** | On any page: dim the room right down, or hold something over the lens. Every capability should say it cannot check, in words and aloud, rather than showing a green all-clear — a scene the AI cannot see is not a safe one |
| **Doors** | *Mark doors* → drag a box around each doorway and name it. Nothing is watched until you do; those doors are then timed, and the allowed open time is on the right |
| **Cameras** | The camera register. Start any camera for the first time and you are asked to name it and place it — once. Alerts then carry the camera's name and location, and this page lists every registered camera with its status and its **camera clock**: ✓ when a timestamp is being read from its pictures, ⚠ when none can be found. A camera with no usable clock raises one warning that stays until a clock appears, and says plainly that its events are being stamped by the system instead — nothing stops watching |
| **Timestamps** | Play a recording with a clock burned into the picture — most CCTV exports have one — and its events are stamped with *that* moment rather than the moment you replayed it: an April recording reviewed today files under April. The camera card says which clock is being read; if the clock sits somewhere unusual, *Mark timestamp area* draws a box over it and the AI reads exactly there. With no readable clock the system's own time is used, and every event says which of the two stamped it |
| **Lab** *(Training section)* | A simulated factory floor running the real pipeline — the picture check, the model's confidence, the safety rules, the sightings an accusation needs before it is raised, and the decision that follows. Drag a worker into the restricted zone and watch it take three agreeing sightings before it alarms; take a helmet off; open the door and watch the alert escalate the longer it stays open; cut the lighting and watch it refuse to judge rather than report a calm floor. **Try This** walks through five of these one at a time, and the tutor answers from the live frame. Nothing here touches a camera, a worker or the event history — the one page where breaking things costs nothing |
| **Registration** *(Training section)* | Register a worker — name, employee ID, designation, and the rest, plus **1–5 photographs (required)**. The photos become the worker's picture on the desk pages and their **Face Recognition enrollment**: cameras recognise registered workers by name, drawn **green, never as an alarm** — the red watchlist behaviour is untouched. The answer is a **link** to hand them; one of three induction programs is allotted at random when you save, and the list below keeps every link recoverable |
| **The worker's link** | Opens on the worker's own phone, with none of this dashboard around it. It runs their allotted program section by section, issues a **certificate** in their name on completion, then offers **Take assessment** — five questions, graded on the server, never sending the answer key to the phone — and ends with a **score card** (pass mark 3/5, retakes allowed). Certificate and score card both save as PDF from the phone. The link resumes wherever the worker left off |
| **Programs** *(Training section)* | The three induction programs — General Safety Induction, Fire Safety & Emergency Response, Machine & Equipment Safety — readable in full, with how many workers each has been allotted to |
| **Assessment** *(Training section)* | The quiz bank: every program's five questions with the correct answer marked — the one surface that shows the key, and it stays on this desk; the worker's phone is only ever graded against it. Beside each quiz, how many workers have taken it and how many passed |
| **Status** *(Training section)* | Every registered worker and where they stand: training completed or pending, their assessment score, and the skill verdict — **Skilled** at 60% or better, **Unskilled** below it, and no verdict at all until they have been assessed. The same verdict appears on the Face Recognition page's identified-worker card when a camera recognises them |
| **Safety Events** | Everything the system has seen, with the picture it saw it in. Each one says which clock stamped it and how that camera's clock stood at the time. Events stamped from old footage sort by their own date, so widen the period to **1 year** to find them |
| **Reports** | Counts by capability and by day, and an export for Excel. The period reaches a year for archive footage; an export stops at its newest 500 rows and the page says so when a filter matches more |

**Known limits**

- **A worker's link is the only key to their training page.** There are no logins anywhere in this POC, so anyone holding the link can act as that worker — hand links to their owners, not to a group chat. Registrations live on this runtime's disk and end with it, like everything else here.
- An RTSP camera has to be reachable from the public internet. A camera on
  your plant LAN will not be reachable from Colab.
- A curfew is judged on the **footage's own burned-in clock** when the source
  has one, and on the wall clock when it does not — the page says which, and
  what that clock reads right now. The recording's clock matters when
  reviewing footage: replaying last night's video this morning flags whoever
  was in the bay at 23:00, rather than asking whether *now* is inside the
  window.
- **A curfew keeps the hours of the browser that saved it**, not Colab's.
  Colab runs on UTC, so a window typed as 22:00 in an Indian control room
  would otherwise run at 03:30 the next morning — the timezone is saved with
  the window to stop exactly that. If the two ever disagree the Curfew panel
  says by how much, and re-saving the window from the machine you are
  reading moves it onto that clock.
- **Suspended load detection is the first phase of that capability.** It
  reports who is standing in the lifting area. Whether a load is present,
  raised or suspended is reported as **not known** — never as "no", because
  the system has no way of knowing yet. Reading a page of that name as an
  all-clear about a hanging load is the misunderstanding most likely to hurt
  somebody, so the page says which of its six questions it can answer.
  Training the detector that would answer the rest needs footage of the bay;
  the dataset rules, and the gate that enforces them, are in
  `training/suspended_load/ANNOTATING.md`.
- That module carries its own person weights (`yolov8m-seg`, about 55 MB of
  the clone) rather than sharing the detector the other modules use. On the
  reference bay footage the shared nano weights returned **nobody** on three
  consecutive samples of frames holding three people, at the normal
  confidence bar; the measured table is in the module's docstring. It costs
  more per frame, which a GPU absorbs and a CPU would not — and it still
  misses a worker standing by the jib arm in one of those frames.
- Colab wipes the machine when it disconnects — including the event history,
  any area you drew and any doors you calibrated. Nothing here survives the
  session yet.
- The forklift weights are trained on one class and were measured on two clips
  of one warehouse: a real forklift scores 0.76–0.84 there and the strongest
  thing that was not one scored 0.41, which is where the 65% alarm setting
  comes from. On unrelated footage the same weights call a worker's forearm a
  forklift at 0.84. They have not been shown what a forklift *is not*, so on a
  site they have not been measured on, check the setting before trusting it —
  the page shows what it found and at what score.

- The walkway detector carries no object model: it learns what the marked
  lane's own floor looks like and reports what is not it. That is why it finds
  a cardboard box nothing was ever trained on — but it is blind to an object
  the same colour as the floor beneath it, it cannot say *what* it has found,
  and it needs the marked lane to be mostly floor. Mark a bay stacked with
  pallets and the pallets become the floor.

- A clock burned into a picture never says what timezone it is in, so it
  is stored and shown exactly as the footage shows it rather than being
  converted to yours. The reading is checked before it is believed: a
  clock that jumps backwards is treated as misread, and one unreadable
  frame is not enough to call a camera clockless.
- The timestamp reader is the one optional install above. Skip it and
  nothing breaks — every event simply carries the system clock, and the
  Cameras page says so.


## 11. Stop everything

In [ ]:
for name in ("tunnel", "server"):
    proc = globals().get(name)
    if proc is None:
        print(name, "was not started")
        continue
    try:
        proc.terminate()
        print(name, "stopped")
    except Exception as e:
        print(name, "->", e)